Small feasibility study: real digits hyperparameter trials, selective duplicate executions, checkpoint retry, and local process loss. No SMT or cloud resources yet. Edit the settings and run in order. Helpers are ordinary Python files in `selective_replication/`.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "gossip_benchmarks").exists():
    ROOT = ROOT.parent
HELPERS = ROOT / "gossip_benchmarks" / "selective_replication"
assert HELPERS.exists()

# Each study gets a new directory. Set this explicitly to inspect existing results.
RUN = ROOT / "ray_replication_results" / datetime.now().strftime("%Y%m%d-%H%M%S")
CONFIG = {
    "machines": {  # Hypothetical hourly costs; NOT live GCP prices.
        "safe": {"cost_per_hour": 1.0},
        "spot_a": {"cost_per_hour": 1.0},
        "spot_b": {"cost_per_hour": 1.0},
    },
    "tasks": {
        f"width-{width}": {
            "width": width, "epochs": 40, "lr": 0.03,
            "seed": 123, "batch_size": 64, "checkpoint_every": 10,
        }
        for width in (128, 256, 512)
    },
    "profile_repeats": 2,
    "failure_machines": ["spot_a", "spot_b"],
    "spot_cost_multipliers": [0.25, 1.0],
    "failure_probabilities": [0.1, 0.5],  # Total probability of a fault scenario.
    "deadline_multipliers": [1.25, 2.0],
    "fault_time_fraction": 0.45,
    "success_target": 0.95,
}
# One CPU per logical machine; model/optimizer/input-state absolute tolerance is 1e-5.
CONFIG_PATH = RUN.with_suffix(".config.json")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open("x") as stream:
    json.dump(CONFIG, stream, indent=2)

def run(stage, *extra):
    subprocess.run([sys.executable, str(HELPERS / "run_study.py"), stage,
                    "--folder", str(RUN), *map(str, extra)], check=True)

print(RUN)

In [ ]:
run("profile", "--config", CONFIG_PATH)

In [ ]:
run("search")
search_result = json.loads((RUN / "search.json").read_text())
for index, case in enumerate(search_result["cases"]):
    print(index, case["name"])
    for policy, result in case["policies"].items():
        print(" ", policy, "INFEASIBLE" if result is None else {
            "modeled_cost": result["expected_cost"],
            "modeled_success": result["success_probability"],
            "plan": result["plan"],
        })

In [ ]:
# None selects up to two cases, including the largest MODELED advantage.
# Set e.g. CASES = [0, 1, 2, 3] to broaden coverage. Replays include a held-out fault time.
CASES = None
REPEATS = 2
HOLDOUT_FRACTION = 0.8
extra = [] if CASES is None else ["--cases", *CASES]
run("replay", "--repeats", REPEATS, "--holdout-fraction", HOLDOUT_FRACTION, *extra)

In [ ]:
from IPython.display import Image, display

subprocess.run([sys.executable, str(HELPERS / "report.py"), str(RUN)], check=True)
if (RUN / "modeled-cost.png").exists():
    display(Image(filename=str(RUN / "modeled-cost.png")))